# 🌐 MBart Many-to-Many Fine-Tuning Notebook
### Bhaasha Translation Models — IDEA Project

**Supported Language Pairs (train one at a time):**
| # | Source → Target | SRC_LANG | TGT_LANG | Dataset CSV columns |
|---|----------------|----------|----------|---------------------|
| 1 | Marathi → English | `mr_IN` | `en_XX` | `marathi`, `english` |
| 2 | Marathi → Hindi | `mr_IN` | `hi_IN` | `marathi`, `hindi` |
| 3 | Hindi → English | `hi_IN` | `en_XX` | `hindi`, `english` |
| 4 | English → Hindi | `en_XX` | `hi_IN` | `english`, `hindi` |
| 5 | Hindi → Marathi | `hi_IN` | `mr_IN` | `hindi`, `marathi` |
| 6 | English → Marathi | `en_XX` | `mr_IN` | `english`, `marathi` |

**Workflow:**
1. Mount Drive & install deps  
2. Set config (language pair + file paths)  
3. Load datasets  
4. Evaluate base model (benchmark BEFORE fine-tuning)  
5. Fine-tune on training data  
6. Evaluate fine-tuned model (benchmark AFTER fine-tuning)  
7. Save model to Drive in repo-compatible format  

## ⚙️ Step 1 — Install Dependencies & Mount Google Drive

In [1]:
# Install required packages
!pip install -q -U transformers datasets sacrebleu sentencepiece accelerate
print("✅ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.0 MB/s eta 0:00:00
✅ Packages installed


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

Mounted at /content/drive
✅ Google Drive mounted at /content/drive


In [3]:
import torch
print(f"🖥️  CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU             : {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — go to Runtime > Change Runtime Type > T4 GPU")

🖥️  CUDA Available : False
⚠️  No GPU found — go to Runtime > Change Runtime Type > T4 GPU


---
## 🔧 Step 2 — Configuration
> **Only edit this cell** when switching language pairs.
> Everything else runs automatically.

In [4]:
# ============================================================
#  ✏️  EDIT ONLY THIS CELL — Language Pair Configuration
# ============================================================

# --- Language codes (MBart format) ---
# Marathi = mr_IN | English = en_XX | Hindi = hi_IN
SRC_LANG = "en_XX"   # Source language code
TGT_LANG = "mr_IN"   # Target language code

# --- Column names in your CSV file ---
# These are the column names for source and target text in your CSV
SRC_COL  = "english"    # Column name for source language text
TGT_COL  = "marathi"    # Column name for target language text

# --- Dataset paths on Google Drive ---
# Update these paths to where your CSVs are on Drive
DRIVE_BASE   = "/content/drive/MyDrive/"          # Base Drive folder
TRAIN_CSV    = f"{DRIVE_BASE}/train-mar-eng.csv"  # Training data (~50k rows)
TEST_CSV     = f"{DRIVE_BASE}/test-mar-eng.csv"   # Test data (~20k rows)
VAL_CSV      = f"{DRIVE_BASE}/val-mar-eng.csv"    # Validation data (~10k rows)

# --- Model config (matches config.py in the repo) ---
BASE_MODEL   = "facebook/mbart-large-50-many-to-many-mmt"  # DO NOT CHANGE

# --- Output path: where to save the fine-tuned model on Drive ---
# Follows repo convention: resources/translation/model/<SrcLang>_<TgtLang>/
lang_name_map = {
    "mr_IN": "Marathi",
    "en_XX": "English",
    "hi_IN": "Hindi"
}
SRC_NAME     = lang_name_map[SRC_LANG]
TGT_NAME     = lang_name_map[TGT_LANG]
MODEL_SAVE_DIR   = f"{DRIVE_BASE}/resources/translation/model/{SRC_NAME}_{TGT_NAME}"
TOKENIZER_SAVE_DIR = f"{DRIVE_BASE}/resources/translation/tokenizer/{SRC_NAME}_{TGT_NAME}"

# --- Training hyperparameters ---
MAX_INPUT_LEN   = 128    # Max tokens for input  (increase to 256 if sentences are long)
MAX_TARGET_LEN  = 128    # Max tokens for output
BATCH_SIZE      = 8      # Per-device batch size (reduce to 4 if OOM error)
GRAD_ACCUM      = 4      # Gradient accumulation steps → effective batch = 8 × 4 = 32
NUM_EPOCHS      = 3      # Fine-tuning epochs
LEARNING_RATE   = 5e-5   # Learning rate
WARMUP_STEPS    = 200    # Warmup steps
EVAL_STEPS      = 500    # Evaluate every N steps
SAVE_STEPS      = 500    # Save checkpoint every N steps
BLEU_SAMPLE_SIZE = 500   # Rows to use for BLEU evaluation (keep low for speed)

# ============================================================
print(f"✅ Config set: {SRC_LANG} ({SRC_NAME}) → {TGT_LANG} ({TGT_NAME})")
print(f"   Train CSV   : {TRAIN_CSV}")
print(f"   Test CSV    : {TEST_CSV}")
print(f"   Val CSV     : {VAL_CSV}")
print(f"   Model save  : {MODEL_SAVE_DIR}")
print(f"   Tokenizer   : {TOKENIZER_SAVE_DIR}")

✅ Config set: en_XX (English) → mr_IN (Marathi)
   Train CSV   : /content/drive/MyDrive//train-mar-eng.csv
   Test CSV    : /content/drive/MyDrive//test-mar-eng.csv
   Val CSV     : /content/drive/MyDrive//val-mar-eng.csv
   Model save  : /content/drive/MyDrive//resources/translation/model/English_Marathi
   Tokenizer   : /content/drive/MyDrive//resources/translation/tokenizer/English_Marathi


---
## 📂 Step 3 — Load Datasets

In [5]:
import pandas as pd

def load_csv(path, src_col, tgt_col):
    """Load CSV and drop rows with missing values in source or target columns."""
    df = pd.read_csv(path)
    print(f"  Loaded  : {path}")
    print(f"  Shape   : {df.shape}")
    print(f"  Columns : {list(df.columns)}")
    # Validate required columns exist
    assert src_col in df.columns, f"❌ Column '{src_col}' not found! Available: {list(df.columns)}"
    assert tgt_col in df.columns, f"❌ Column '{tgt_col}' not found! Available: {list(df.columns)}"
    # Drop empty rows
    df = df[[src_col, tgt_col]].dropna()
    df[src_col] = df[src_col].astype(str).str.strip()
    df[tgt_col] = df[tgt_col].astype(str).str.strip()
    df = df[(df[src_col] != '') & (df[tgt_col] != '')]
    print(f"  Clean rows: {len(df)}")
    return df

print("📂 Loading datasets...")
train_df = load_csv(TRAIN_CSV, SRC_COL, TGT_COL)
print()
test_df  = load_csv(TEST_CSV,  SRC_COL, TGT_COL)
print()
val_df   = load_csv(VAL_CSV,   SRC_COL, TGT_COL)

print(f"\n✅ Dataset sizes → Train: {len(train_df)} | Test: {len(test_df)} | Val: {len(val_df)}")
print("\n📝 Sample data:")
train_df.head(3)

📂 Loading datasets...
  Loaded  : /content/drive/MyDrive//train-mar-eng.csv
  Shape   : (50000, 2)
  Columns : ['english', 'marathi']
  Clean rows: 50000

  Loaded  : /content/drive/MyDrive//test-mar-eng.csv
  Shape   : (20000, 2)
  Columns : ['english', 'marathi']
  Clean rows: 20000

  Loaded  : /content/drive/MyDrive//val-mar-eng.csv
  Shape   : (10000, 2)
  Columns : ['english', 'marathi']
  Clean rows: 10000

✅ Dataset sizes → Train: 50000 | Test: 20000 | Val: 10000

📝 Sample data:


,english,marathi
0,Punjab chief minister Amarinder Singh had also...,पंजाबचे मुख्यमंत्री कॅप्टन अमरिंदर सिंह यांनीह...
1,They have also demanded an apology from chief ...,यावेळी त्यांनी देवेंद्र फडणवीस यांच्या राजीनाम...
2,The State Government has asked...,अशी विचारणा राज्य सरकारला केली .


---
## 🤖 Step 4 — Load Base Model & Tokenizer
> Loading `facebook/mbart-large-50-many-to-many-mmt` — same as the repo's `config.py`

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50Tokenizer
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️  Loading base model: {BASE_MODEL}")
print(f"🖥️  Device: {device}")

# Load tokenizer — sets source language (same as download_helper.py in the repo)
tokenizer = MBart50Tokenizer.from_pretrained(
    BASE_MODEL,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG
)
print(f"✅ Tokenizer loaded — src_lang={SRC_LANG}, tgt_lang={TGT_LANG}")

# Load base model
base_model = MBartForConditionalGeneration.from_pretrained(BASE_MODEL)
base_model = base_model.to(device)
print(f"✅ Base model loaded and moved to {device}")
print(f"📊 Model parameters: {sum(p.numel() for p in base_model.parameters()):,}")

⚙️  Loading base model: facebook/mbart-large-50-many-to-many-mmt
🖥️  Device: cpu


---
## 📊 Step 5 — Evaluate Base Model (Benchmark BEFORE Fine-tuning)
> This gives you the starting BLEU score to compare against after fine-tuning

In [ ]:
import sacrebleu
from tqdm import tqdm

def evaluate_bleu(model, tokenizer, df, src_col, tgt_col, src_lang, tgt_lang,
                  sample_size=500, batch_size=8, max_length=128):
    """
    Compute BLEU score on a sample of the test data.
    Uses the same translation logic as translator.py in the repo.
    Returns: bleu_score, predictions, references, sample_df
    """
    model.eval()
    # Sample rows for evaluation speed — save sample_df to return it
    sample_df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)

    all_predictions = []
    all_references  = []

    for i in tqdm(range(0, len(sample_df), batch_size), desc="Evaluating"):
        batch = sample_df.iloc[i : i + batch_size]
        src_texts = batch[src_col].tolist()
        tgt_texts = batch[tgt_col].tolist()

        inputs = tokenizer(
            src_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )

        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend(preds)
        all_references.extend(tgt_texts)

    bleu = sacrebleu.corpus_bleu(all_predictions, [all_references])
    # ✅ Also return sample_df so display uses the correct (shuffled) rows
    return bleu.score, all_predictions, all_references, sample_df


print(f"📊 Evaluating BASE MODEL on {BLEU_SAMPLE_SIZE} test samples...")
print(f"   {SRC_LANG} ({SRC_NAME}) → {TGT_LANG} ({TGT_NAME})")
print("-" * 50)

base_bleu, base_preds, base_refs, base_sample_df = evaluate_bleu(
    base_model, tokenizer, test_df,
    SRC_COL, TGT_COL, SRC_LANG, TGT_LANG,
    sample_size=BLEU_SAMPLE_SIZE
)

print(f"\n🎯 BASE MODEL BLEU Score: {base_bleu:.2f}")
print("\n📝 Sample Translations (Base Model):")
for i in range(min(5, len(base_preds))):
    # ✅ Use base_sample_df instead of test_df.iloc[i] — same shuffled order as refs/preds
    print(f"  [{i+1}] Source    : {base_sample_df.iloc[i][SRC_COL][:80]}")
    print(f"       Reference : {base_refs[i][:80]}")
    print(f"       Predicted : {base_preds[i][:80]}")
    print()

---
## 🔄 Step 6 — Tokenize & Prepare Data for Training

In [ ]:
from datasets import Dataset

def tokenize_batch(examples, tokenizer, src_col, tgt_col, src_lang, tgt_lang,
                   max_input_len, max_target_len):
    """
    Tokenize source and target texts for Seq2Seq training.
    Fixed for transformers >= 4.39 — as_target_tokenizer() was removed.
    """
    tokenizer.src_lang = src_lang

    # ✅ New API: pass text_target directly instead of using as_target_tokenizer()
    model_inputs = tokenizer(
        examples[src_col],
        text_target=examples[tgt_col],
        max_length=max_input_len,
        max_target_length=max_target_len,
        padding="max_length",
        truncation=True,
        return_tensors=None   # Return lists, not tensors (datasets handles this)
    )

    # Replace padding token ids in labels with -100 (ignored by loss)
    model_inputs["labels"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in model_inputs["labels"]
    ]

    return model_inputs


print("🔄 Converting DataFrames to HuggingFace Datasets...")
train_dataset = Dataset.from_pandas(train_df[[SRC_COL, TGT_COL]].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[[SRC_COL, TGT_COL]].reset_index(drop=True))
print(f"   Train: {len(train_dataset)} examples")
print(f"   Val  : {len(val_dataset)} examples")

print("\n⚙️  Tokenizing datasets (this may take a few minutes)...")
tokenized_train = train_dataset.map(
    lambda examples: tokenize_batch(
        examples, tokenizer, SRC_COL, TGT_COL, SRC_LANG, TGT_LANG,
        MAX_INPUT_LEN, MAX_TARGET_LEN
    ),
    batched=True,
    batch_size=64,
    remove_columns=train_dataset.column_names,   # ✅ Remove original text columns
    desc="Tokenizing train"
)

tokenized_val = val_dataset.map(
    lambda examples: tokenize_batch(
        examples, tokenizer, SRC_COL, TGT_COL, SRC_LANG, TGT_LANG,
        MAX_INPUT_LEN, MAX_TARGET_LEN
    ),
    batched=True,
    batch_size=64,
    remove_columns=val_dataset.column_names,     # ✅ Remove original text columns
    desc="Tokenizing val"
)

# Set format for PyTorch
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format(type="torch",   columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization complete")
print(f"   Tokenized train features : {tokenized_train.features}")
print(f"   Tokenized train columns  : {tokenized_train.column_names}")

---
## 🏋️ Step 7 — Fine-Tune the Model
> Uses HuggingFace `Seq2SeqTrainer` with GPU acceleration

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import numpy as np
import os

# Create output dir for checkpoints
CHECKPOINT_DIR = f"/content/checkpoints/{SRC_NAME}_{TGT_NAME}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Data collator — handles dynamic padding during training
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=base_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

# Training arguments — optimized for Colab T4 GPU
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=torch.cuda.is_available(),
    # ✅ logging_dir removed — deprecated in v5.2
    logging_steps=100,
    report_to="none",
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
)

# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=base_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print(f"🏋️  Starting fine-tuning: {SRC_NAME} → {TGT_NAME}")
print(f"   Epochs       : {NUM_EPOCHS}")
print(f"   Train samples: {len(tokenized_train)}")
print(f"   Batch size   : {BATCH_SIZE} × {GRAD_ACCUM} accum = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   FP16 (mixed) : {torch.cuda.is_available()}")
print("-" * 50)

# ---- START TRAINING ----
train_result = trainer.train()

print("\n✅ Fine-tuning complete!")
print(f"   Training loss : {train_result.training_loss:.4f}")
print(f"   Total steps   : {train_result.global_step}")
print(f"   Training time : {train_result.metrics.get('train_runtime', 0):.0f} seconds")

---
## Step 8 — Evaluate Fine-Tuned Model (Benchmark AFTER Fine-tuning)
> Compare this BLEU score with the base model score from Step 5

In [ ]:
# Get the best model (loaded automatically by trainer because load_best_model_at_end=True)
finetuned_model = trainer.model
finetuned_model.eval()

print(f"📊 Evaluating FINE-TUNED MODEL on {BLEU_SAMPLE_SIZE} test samples...")
print(f"   {SRC_LANG} ({SRC_NAME}) → {TGT_LANG} ({TGT_NAME})")
print("-" * 50)

# ✅ Unpack 4 values now (added ft_sample_df)
ft_bleu, ft_preds, ft_refs, ft_sample_df = evaluate_bleu(
    finetuned_model, tokenizer, test_df,
    SRC_COL, TGT_COL, SRC_LANG, TGT_LANG,
    sample_size=BLEU_SAMPLE_SIZE
)

# ---- RESULTS SUMMARY ----
print("\n" + "="*55)
print("       📈  BLEU SCORE COMPARISON")
print("="*55)
print(f"  Language Pair   : {SRC_NAME} ({SRC_LANG}) → {TGT_NAME} ({TGT_LANG})")
print(f"  Eval samples    : {BLEU_SAMPLE_SIZE}")
print("-"*55)
print(f"  Base Model BLEU : {base_bleu:.2f}")
print(f"  Fine-tuned BLEU : {ft_bleu:.2f}")
improvement = ft_bleu - base_bleu
emoji = "🚀" if improvement > 0 else "⚠️"
print(f"  Improvement     : {emoji} {improvement:+.2f} BLEU points")
print("="*55)

print("\n📝 Sample Translations (Fine-tuned Model):")
for i in range(min(5, len(ft_preds))):
    # ✅ Use ft_sample_df to ensure Source matches the Reference/Prediction
    print(f"  [{i+1}] Source    : {ft_sample_df.iloc[i][SRC_COL][:80]}")
    print(f"       Reference : {ft_refs[i][:80]}")
    print(f"       Predicted : {ft_preds[i][:80]}")
    print()

---
## 💾 Step 9 — Save Fine-Tuned Model to Google Drive
> Saves in the same folder structure as the repo's `resources/translation/model/` and `resources/translation/tokenizer/`

In [ ]:
import os

os.makedirs(MODEL_SAVE_DIR,    exist_ok=True)
os.makedirs(TOKENIZER_SAVE_DIR, exist_ok=True)

print(f"💾 Saving fine-tuned model to Drive...")
print(f"   Model     → {MODEL_SAVE_DIR}")
print(f"   Tokenizer → {TOKENIZER_SAVE_DIR}")

# Save model — compatible with how the repo loads it
finetuned_model.save_pretrained(MODEL_SAVE_DIR)
print("   ✅ Model saved")

# Save tokenizer — compatible with how translator.py loads it
tokenizer.save_pretrained(TOKENIZER_SAVE_DIR)
print("   ✅ Tokenizer saved")

# Verify saved files
model_files = os.listdir(MODEL_SAVE_DIR)
tok_files   = os.listdir(TOKENIZER_SAVE_DIR)
print(f"\n📁 Model files saved     : {model_files}")
print(f"📁 Tokenizer files saved  : {tok_files}")
print(f"\n🎉 Done! Model for {SRC_NAME} → {TGT_NAME} is ready.")

---
## 🧪 Step 10 — Quick Inference Test
> Test your saved model on a custom sentence

In [ ]:
# ✏️ Edit this sentence to test
TEST_SENTENCE = "अर्जेंटीना के स्ट्राइकर को शुरुआती दौर में ही"   # Change this to any source language sentence

def translate_sentence(model, tokenizer, text, src_lang, tgt_lang, max_length=128):
    """
    Translate a single sentence.
    Matches the translate() method in translator.py exactly.
    """
    model.eval()
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
    translated = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return translated

result = translate_sentence(finetuned_model, tokenizer, TEST_SENTENCE, SRC_LANG, TGT_LANG)

print("🧪 Quick Inference Test")
print(f"   Input  ({SRC_NAME}): {TEST_SENTENCE}")
print(f"   Output ({TGT_NAME}): {result}")

In [ ]:
# 1. EVALUATE TRAINED MODEL
print(f"📊 Evaluating FINE-TUNED MODEL on {BLEU_SAMPLE_SIZE} test samples...")
print(f"   {SRC_LANG} ({SRC_NAME}) → {TGT_LANG} ({TGT_NAME})")
print("-" * 50)

# ✅ Unpack 4 values (ft_bleu, ft_preds, ft_refs, ft_sample_df)
ft_bleu, ft_preds, ft_refs, ft_sample_df = evaluate_bleu(
    finetuned_model, tokenizer, test_df,
    SRC_COL, TGT_COL, SRC_LANG, TGT_LANG,
    sample_size=BLEU_SAMPLE_SIZE
)

# 2. SHOW COMPARISON
print("\n" + "="*55)
print("       📈  BLEU SCORE COMPARISON")
print("="*55)
print(f"  Base Model BLEU : {base_bleu:.2f}")
print(f"  Fine-tuned BLEU : {ft_bleu:.2f}")
improvement = ft_bleu - base_bleu
emoji = "🚀" if improvement > 0 else "⚠️"
print(f"  Improvement     : {emoji} {improvement:+.2f} BLEU points")
print("="*55)

# 3. SHOW SAMPLES FROM TRAINED MODEL
print("\n📝 Sample Translations (Fine-tuned Model):")
for i in range(min(5, len(ft_preds))):
    print(f"  [{i+1}] Source    : {ft_sample_df.iloc[i][SRC_COL][:80]}")
    print(f"       Reference : {ft_refs[i][:80]}")
    print(f"       Predicted : {ft_preds[i][:80]}")
    print()

# 4. QUICK INFERENCE TEST
print("\n🧪 Quick Inference Test on Custom Sentence")
print("-" * 50)
result = translate_sentence(finetuned_model, tokenizer, TEST_SENTENCE, SRC_LANG, TGT_LANG)
print(f"   Input  ({SRC_NAME}): {TEST_SENTENCE}")
print(f"   Output ({TGT_NAME}): {result}")

---
## 📋 Notes for Other Language Pairs

To train the next language pair, go back to **Step 2 (Config Cell)** and update:

```python
# Marathi → Hindi
SRC_LANG = "mr_IN"
TGT_LANG = "hi_IN"
SRC_COL  = "marathi"
TGT_COL  = "hindi"
TRAIN_CSV = "/content/drive/MyDrive/train-marathi-hindi.csv"
...

# Hindi → English
SRC_LANG = "hi_IN"
TGT_LANG = "en_XX"
SRC_COL  = "hindi"
TGT_COL  = "english"
TRAIN_CSV = "/content/drive/MyDrive/train-hindi.csv"
...

# English → Hindi
SRC_LANG = "en_XX"
TGT_LANG = "hi_IN"
SRC_COL  = "english"
TGT_COL  = "hindi"
...

# Hindi → Marathi
SRC_LANG = "hi_IN"
TGT_LANG = "mr_IN"
SRC_COL  = "hindi"
TGT_COL  = "marathi"
...

# English → Marathi
SRC_LANG = "en_XX"
TGT_LANG = "mr_IN"
SRC_COL  = "english"
TGT_COL  = "marathi"
...
```

Then **Run All Cells** (`Runtime > Run All`).

---
### 📁 Saved Model Folder Structure (matches repo's `config.py`)
```
MyDrive/
└── resources/
    └── translation/
        ├── model/
        │   ├── Marathi_English/     ← fine-tuned model files
        │   ├── Marathi_Hindi/
        │   ├── Hindi_English/
        │   ├── English_Hindi/
        │   ├── Hindi_Marathi/
        │   └── English_Marathi/
        └── tokenizer/
            ├── Marathi_English/     ← tokenizer files
            ├── Marathi_Hindi/
            └── ...
```
This matches `save_dir = 'resources/translation'` and `bart_lang_dict` in `config.py`.

---
### ⚠️ If you hit OOM (Out of Memory) on GPU:
- Reduce `BATCH_SIZE` from 8 → 4
- Increase `GRAD_ACCUM` from 4 → 8 to compensate
- Reduce `MAX_INPUT_LEN` from 128 → 64

### 🕐 Approximate training times on T4 GPU:
| Dataset size | Estimated time (3 epochs) |
|-------------|---------------------------|
| 10k rows    | ~30–45 min |
| 50k rows    | ~2.5–3 hours |
| 80k rows    | ~4–5 hours |